In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("yandex/yambda", repo_type="dataset")
print([f for f in files if "50m" in f])

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download("yandex/yambda", "flat/50m/listens.parquet", repo_type="dataset")
listens = pd.read_parquet(path)
print(listens.shape)
print(listens.head())

In [ ]:
print("Пользователей:", listens["uid"].nunique())
print("Треков:", listens["item_id"].nunique())
print("Событий:", len(listens))

In [ ]:
print("Min timestamp:", listens["timestamp"].min())
print("Max timestamp:", listens["timestamp"].max())

In [ ]:
import matplotlib.pyplot as plt

# Сколько событий прослушивания у каждого пользователя
user_counts = listens["uid"].value_counts()

# Сколько событий приходится на каждый трек
track_counts = listens["item_id"].value_counts()

print(user_counts.describe())
print(track_counts.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_counts.hist(bins=100, ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Прослушивания на пользователя")
axes[0].set_xlabel("Количество прослушиваний")
axes[0].set_ylabel("Количество пользователей")

track_counts.hist(bins=100, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Прослушивания на трек")
axes[1].set_xlabel("Количество прослушиваний")
axes[1].set_ylabel("Количество треков")



plt.tight_layout()
plt.show()



In [ ]:
organic_share = (listens["is_organic"] == 1).mean()
print(f"Доля органических прослушиваний: {organic_share:.2%}")

In [ ]:
r = listens["played_ratio_pct"]

print("Прослушивание < 10%:", (r < 10).mean().round(4))
print("Прослушивание < 20%:", (r < 20).mean().round(4))
print("Полные прослушивания:", (r == 100).mean().round(4))
print("Пропуски:", r.isna().mean().round(4))

In [ ]:
DAY = 86400          # проверь по max(timestamp)!
GAP = 30 * 60

t_max = listens["timestamp"].max()
test_start = t_max - DAY
train_end  = test_start - GAP

# сигнал "понравилось": прослушано не менее 50% трека
a = listens["played_ratio_pct"] >= 50
pos = listens[a]

train = pos[pos["timestamp"] < train_end]
test  = pos[pos["timestamp"] >= test_start]



In [ ]:
import numpy as np

def recall_at_k(recs, truth, k=100):
    # recs: dict uid -> список item_id по убыванию score
    # truth: dict uid -> set of item_id из теста
    scores = []
    for uid, t in truth.items():
        hits = len(set(recs.get(uid, [])[:k]) & t)
        scores.append(hits / len(t))
    return np.mean(scores)

def ndcg_at_k(recs, truth, k=100):
    scores = []
    for uid, t in truth.items():
        r = recs.get(uid, [])[:k]
        dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(r) if it in t)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(t), k)))
        scores.append(dcg / idcg)
    return np.mean(scores)